In [5]:
import copy
from heapq import heappush, heappop

# Bài tập yêu cầu n > 4. Ở đây ta gán n = 5 (tương đương 24-puzzle).
# Bạn có thể đổi n = 4 nếu muốn giải 15-puzzle.
n = 5

# 4 vị trí dịch chuyển tương ứng bottom, left, top, right
rows = [ 1, 0, -1, 0 ]
cols = [ 0, -1, 0, 1 ]

# Tạo một lớp hàng đợi
class priorityQueue:
    def __init__(self):
        self.heap = []

    # Inserting a new key 'key'
    def push(self, key):
        heappush(self.heap, key)

    # funct to remove the element that is min from the Priority Queue
    def pop(self):
        return heappop(self.heap)

    def is_empty(self):
        return len(self.heap) == 0

# structure of the node
class nodes:
    def __init__(self, parent, mats, empty_tile_posi, costs, levels):
        self.parent = parent

        # Useful for Storing the matrix
        self.mats = mats

        self.empty_tile_posi = empty_tile_posi
        self.costs = costs
        self.levels = levels

    def __lt__(self, nxt):
        return self.costs < nxt.costs

# Tính h(n): Số ô sai vị trí (Hamming distance)
def calculateCosts(mats, final) -> int:
    count = 0
    for i in range(n):
        for j in range(n):
            if ((mats[i][j]) and (mats[i][j] != final[i][j])):
                count += 1
    return count

def newNodes(mats, empty_tile_posi, new_empty_tile_posi, levels, parent, final) -> nodes:
    # Copying data from the parent matrixes to the present matrixes
    new_mats = copy.deepcopy(mats)

    # Lấy tọa độ ô trống hiện tại và tọa độ ô trống mới
    x1, y1 = empty_tile_posi
    x2, y2 = new_empty_tile_posi

    # Đổi chỗ ô trống
    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    # Tính f(n) = g(n) + h(n)
    # levels là g(n) (số bước đã đi), calculateCosts là h(n) (ước lượng đến đích)
    costs = calculateCosts(new_mats, final) + levels

    return nodes(parent, new_mats, new_empty_tile_posi, costs, levels)

# Hàm in ma trận
def printMatrix(mats):
    for i in range(n):
        for j in range(n):
            print("%3d " % (mats[i][j]), end=" ")
        print()
    print()

# Kiểm tra xem tọa độ (x, y) có hợp lệ trên bảng không
def isSafe(x, y):
    return x >= 0 and x < n and y >= 0 and y < n

# Vòng lặp chính của thuật toán A* (AKT)
def solve(initial, empty_tile_posi, final):
    pq = priorityQueue()

    # Tính cost cho root node
    costs = calculateCosts(initial, final)
    root = nodes(None, initial, empty_tile_posi, costs, 0)

    pq.push(root)

    while not pq.is_empty():
        minimum = pq.pop()

        # Nếu node có h(n) = 0 tức là đã đến đích (costs - levels == h(n))
        if minimum.costs - minimum.levels == 0:
            # Truy vết và in đường đi
            path = []
            curr = minimum
            while curr:
                path.append(curr.mats)
                curr = curr.parent
            print(f"--- TÌM THẤY ĐƯỜNG ĐI SAU {minimum.levels} BƯỚC ---")
            for p in reversed(path):
                printMatrix(p)
            return

        # Sinh các trạng thái kề
        x, y = minimum.empty_tile_posi
        for i in range(4):
            new_x = x + rows[i]
            new_y = y + cols[i]

            if isSafe(new_x, new_y):
                # levels + 1 vì tiến thêm 1 bước
                child = newNodes(minimum.mats, minimum.empty_tile_posi, (new_x, new_y), minimum.levels + 1, minimum, final)
                pq.push(child)

# ==========================================
# KHU VỰC CHẠY THỬ (TESTING) VỚI N = 5
# ==========================================

# Ma trận bắt đầu (0 là ô trống)
initial_state = [
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18, 19, 20],
    [21, 22, 23, 0, 24] # Ô trống ở vị trí (4, 3)
]

# Ma trận đích
final_state = [
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18, 19, 20],
    [21, 22, 23, 24, 0] # Ô trống ở vị trí (4, 4)
]

# Tọa độ ô trống ban đầu trong initial_state
empty_tile_pos = (4, 3)

solve(initial_state, empty_tile_pos, final_state)

--- TÌM THẤY ĐƯỜNG ĐI SAU 1 BƯỚC ---
  1    2    3    4    5  
  6    7    8    9   10  
 11   12   13   14   15  
 16   17   18   19   20  
 21   22   23    0   24  

  1    2    3    4    5  
  6    7    8    9   10  
 11   12   13   14   15  
 16   17   18   19   20  
 21   22   23   24    0  

